Used for descriptive statistics of distributions of latin square counterbalancing

In [12]:
from __future__ import annotations

import csv
from pathlib import Path


TEXTS = ("Bees", "Hubble", "Salt")
CONDITIONS = ("none", "full", "eyetracked")


def _to_float(value: str) -> float | None:
	text = (value or "").strip()
	if not text:
		return None
	try:
		return float(text)
	except ValueError:
		return None


def _mean(values: list[float]) -> float | None:
	if not values:
		return None
	return sum(values) / len(values)


def _fmt(value: float | None) -> str:
	if value is None:
		return "N/A"
	return f"{value:.3f}"

Section for questionnaire per reading condition, eye strain

In [13]:
import pandas as pd
from IPython.display import display

csv_path = Path("aggregated_questionnaire_metrics.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"Could not find questionnaire CSV: {csv_path}")

with csv_path.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)

eye_all: list[float] = []
nasa_all: list[float] = []
eye_by_text: dict[str, list[float]] = {text: [] for text in TEXTS}
eye_by_condition: dict[str, list[float]] = {condition: [] for condition in CONDITIONS}
nasa_by_text: dict[str, list[float]] = {text: [] for text in TEXTS}

for row in rows:
    condition_by_text: dict[str, str] = {}
    for text in TEXTS:
        condition_col = f'Text filtering condition "{text}"'
        condition_value = (row.get(condition_col, "") or "").strip().lower()
        if condition_value in CONDITIONS:
            condition_by_text[text] = condition_value

    for key, raw in row.items():
        if key is None:
            continue

        val = _to_float(raw)
        if val is None:
            continue

        if key.startswith('Text questionnaire ') and '"' in key:
            eye_all.append(val)
            for text in TEXTS:
                if f'"{text}"' in key:
                    eye_by_text[text].append(val)
                    condition = condition_by_text.get(text)
                    if condition:
                        eye_by_condition[condition].append(val)
                    break

        if key.startswith('NASA TLX overall ') and '"' in key:
            nasa_all.append(val)
            for text in TEXTS:
                if f'"{text}"' in key:
                    nasa_by_text[text].append(val)
                    break

overall_table = pd.DataFrame([
    {
        "Metric": "Eye score",
        "Average": _mean(eye_all),
    },
    {
        "Metric": "NASA TLX score",
        "Average": _mean(nasa_all),
    },
])

eye_by_text_table = pd.DataFrame(
    [{"Text": text, "Average eye score": _mean(eye_by_text[text])} for text in TEXTS]
 )

eye_by_condition_table = pd.DataFrame(
    [{"Condition": condition, "Average eye score": _mean(eye_by_condition[condition])} for condition in CONDITIONS]
 )

nasa_by_text_table = pd.DataFrame(
    [{"Text": text, "Average NASA TLX score": _mean(nasa_by_text[text])} for text in TEXTS]
 )

overall_table["Average"] = overall_table["Average"].round(3)
eye_by_text_table["Average eye score"] = eye_by_text_table["Average eye score"].round(3)
eye_by_condition_table["Average eye score"] = eye_by_condition_table["Average eye score"].round(3)
nasa_by_text_table["Average NASA TLX score"] = nasa_by_text_table["Average NASA TLX score"].round(3)

display(overall_table)
display(eye_by_text_table)
display(eye_by_condition_table)
display(nasa_by_text_table)

,Metric,Average
0,Eye score,21.991
1,NASA TLX score,5.759


,Text,Average eye score
0,Bees,19.713
1,Hubble,22.009
2,Salt,24.250


,Condition,Average eye score
0,none,19.713
1,full,22.009
2,eyetracked,24.250


,Text,Average NASA TLX score
0,Bees,5.875
1,Hubble,6.180
2,Salt,5.222


Check averages of eye events per reading section

In [14]:
import pandas as pd
from IPython.display import display

reading_csv_path = Path("aggregated_reading_eye_metrics.csv")
if not reading_csv_path.exists():
    raise FileNotFoundError(f"Could not find reading CSV: {reading_csv_path}")

with reading_csv_path.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    reading_rows = list(reader)

event_columns = [
    "Saccade count",
    "Saccade avg duration",
    "Fixation count",
    "Fixation avg duration",
    "Blink count",
    "Blink avg duration",
]

by_section: dict[str, dict[str, list[float]]] = {}
by_condition: dict[str, dict[str, list[float]]] = {
    condition: {column: [] for column in event_columns}
    for condition in CONDITIONS
}

for row in reading_rows:
    section = (row.get("Reading paragraph", "") or "").strip()
    condition = (row.get("Condition", "") or "").strip().lower()

    if section:
        section_bucket = by_section.setdefault(
            section,
            {column: [] for column in event_columns},
        )
        for column in event_columns:
            value = _to_float(row.get(column, ""))
            if value is not None:
                section_bucket[column].append(value)

    if condition in by_condition:
        condition_bucket = by_condition[condition]
        for column in event_columns:
            value = _to_float(row.get(column, ""))
            if value is not None:
                condition_bucket[column].append(value)

section_rows: list[dict[str, object]] = []
for section in sorted(by_section):
    row_out: dict[str, object] = {"Reading section": section}
    for column in event_columns:
        row_out[column] = _mean(by_section[section][column])
    section_rows.append(row_out)

condition_rows: list[dict[str, object]] = []
for condition in CONDITIONS:
    row_out = {"Condition": condition}
    for column in event_columns:
        row_out[column] = _mean(by_condition[condition][column])
    condition_rows.append(row_out)

section_table = pd.DataFrame(section_rows)
condition_table = pd.DataFrame(condition_rows)

numeric_cols_section = [c for c in section_table.columns if c != "Reading section"]
numeric_cols_condition = [c for c in condition_table.columns if c != "Condition"]
section_table[numeric_cols_section] = section_table[numeric_cols_section].round(3)
condition_table[numeric_cols_condition] = condition_table[numeric_cols_condition].round(3)

display(section_table)
display(condition_table)

,Reading section,Saccade count,Saccade avg duration,Fixation count,Fixation avg duration,Blink count,Blink avg duration
0,Bees,361.857,38.736,238.643,110.827,15.786,216.192
1,Hubble,441.583,39.244,307.083,116.069,19.500,252.042
2,Salt,583.714,38.356,246.286,107.418,15.857,221.843


,Condition,Saccade count,Saccade avg duration,Fixation count,Fixation avg duration,Blink count,Blink avg duration
0,none,361.857,38.736,238.643,110.827,15.786,216.192
1,full,441.583,39.244,307.083,116.069,19.500,252.042
2,eyetracked,583.714,38.356,246.286,107.418,15.857,221.843
